In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_censored
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

base   = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {device}")
print("All imports successful ✅")

Device: cpu
All imports successful ✅


In [2]:
# Load full expression for pathway scoring
expr_full = pd.read_csv(f'{base}/data/processed/expression_full_478.csv', index_col=0)
immune    = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)
clinical  = pd.read_csv(f'{base}/data/processed/clinical_survival.csv', index_col=0)

# Load Hallmark gene sets
import urllib.request, os

hallmark_path = f'{base}/data/external/hallmark_gene_sets.gmt'

def parse_gmt(filepath):
    pathways = {}
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            pathways[parts[0]] = parts[2:]
    return pathways

hallmark = parse_gmt(hallmark_path)

def compute_pathway_scores(expr_df, hallmark_dict):
    scores = {}
    for pathway_name, gene_list in hallmark_dict.items():
        common_genes = [g for g in gene_list if g in expr_df.columns]
        if len(common_genes) >= 5:
            scores[pathway_name] = expr_df[common_genes].mean(axis=1)
        else:
            scores[pathway_name] = pd.Series(0.0, index=expr_df.index)
    return pd.DataFrame(scores)

print("Computing pathway scores...")
pathway_scores = compute_pathway_scores(expr_full, hallmark)

# Align all data
common = pathway_scores.index.intersection(
         immune.index).intersection(clinical.index)

pathway_scores = pathway_scores.loc[common]
immune         = immune.loc[common]
clinical       = clinical.loc[common]

# Clinical features
age           = clinical[['age']].copy()
gender        = (clinical['gender'] == 'male').astype(float).to_frame()
stage_dummies = pd.get_dummies(clinical['stage_group'], prefix='stage')
stage_dummies = stage_dummies.drop(columns=['stage_Stage I'], errors='ignore')
clinical_features = pd.concat([age, gender, stage_dummies],
                               axis=1).astype(float).fillna(0)

# Immune ratios (Sonica suggestion 3)
immune_ratios = pd.DataFrame({
    'CD8_Treg_ratio':  (immune['T cells CD8'] /
                       (immune['T cells regulatory (Tregs)'] + 1e-8)),
    'M1_M2_ratio':     (immune['Macrophages M1'] /
                       (immune['Macrophages M2'] + 1e-8)),
    'NK_act_ratio':    (immune['NK cells activated'] /
                       (immune['NK cells resting'] + 1e-8)),
    'CD8_M2_ratio':    (immune['T cells CD8'] /
                       (immune['Macrophages M2'] + 1e-8)),
}, index=common)

# Survival labels
y = np.array(
    [(bool(e), t) for e, t in zip(clinical['event'], clinical['survival_time'])],
    dtype=[('event', bool), ('time', float)])

print(f"Patients:         {len(common)}")
print(f"Events:           {y['event'].sum()} ({y['event'].mean()*100:.1f}%)")
print(f"Pathway scores:   {pathway_scores.shape[1]}")
print(f"Immune features:  {immune.shape[1]}")
print(f"Immune ratios:    {immune_ratios.shape[1]}")
print(f"Clinical:         {clinical_features.shape[1]}")

Computing pathway scores...
Patients:         478
Events:           121 (25.3%)
Pathway scores:   50
Immune features:  22
Immune ratios:    4
Clinical:         5


In [3]:
class DeepSupervisionFusion(nn.Module):
    """
    Fusion model with deep supervision.
    
    Each stream has its own encoder AND its own auxiliary risk head.
    The auxiliary heads provide gradient signal directly to each encoder
    during training, preventing gradient dilution.
    
    Total loss = Final fusion loss
               + 0.2 × Expression auxiliary loss
               + 0.2 × Immune auxiliary loss  
               + 0.2 × Clinical auxiliary loss
    
    Architecture:
      Pathways (50)  → Encoder → 32 dims → Aux Head → aux_risk_1
      Immune (26)    → Encoder → 32 dims → Aux Head → aux_risk_2
      Clinical (5)   → Encoder → 32 dims → Aux Head → aux_risk_3
                              ↓
                        Attention Fusion → 32 dims → Final Risk
    """
    def __init__(self, pathway_dim=50, immune_dim=26,
                 clinical_dim=5, dropout=0.5):
        super().__init__()

        # ── Encoders ──────────────────────────────────────────────
        self.encoder_pathway = nn.Sequential(
            nn.Linear(pathway_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32)
        )
        self.encoder_immune = nn.Sequential(
            nn.Linear(immune_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 32)
        )
        self.encoder_clinical = nn.Sequential(
            nn.Linear(clinical_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32)
        )

        # ── Auxiliary risk heads (deep supervision) ────────────────
        # Each head takes a 32-dim stream summary → scalar risk
        self.aux_head_pathway  = nn.Linear(32, 1)
        self.aux_head_immune   = nn.Linear(32, 1)
        self.aux_head_clinical = nn.Linear(32, 1)

        # ── Attention over 3 streams ───────────────────────────────
        self.attention = nn.Sequential(
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1)
        )

        # ── Final risk output ──────────────────────────────────────
        self.output = nn.Linear(32, 1)

    def forward(self, x_pathway, x_immune, x_clinical):
        # Encode each stream
        h_pathway  = self.encoder_pathway(x_pathway)   # (batch, 32)
        h_immune   = self.encoder_immune(x_immune)     # (batch, 32)
        h_clinical = self.encoder_clinical(x_clinical) # (batch, 32)

        # Auxiliary risk predictions (deep supervision)
        aux_risk_pathway  = self.aux_head_pathway(h_pathway)   # (batch, 1)
        aux_risk_immune   = self.aux_head_immune(h_immune)     # (batch, 1)
        aux_risk_clinical = self.aux_head_clinical(h_clinical) # (batch, 1)

        # Attention fusion
        streams      = torch.stack([h_pathway, h_immune, h_clinical], dim=1)
        attn_weights = torch.softmax(self.attention(streams), dim=1)
        fused        = (attn_weights * streams).sum(dim=1)

        # Final risk
        final_risk = self.output(fused)

        return final_risk, aux_risk_pathway, aux_risk_immune, \
               aux_risk_clinical, attn_weights.squeeze(-1)


# ── Test architecture ──────────────────────────────────────────────
model_test = DeepSupervisionFusion(
    pathway_dim=50, immune_dim=26, clinical_dim=5).to(device)

x_p = torch.randn(4, 50).to(device)
x_i = torch.randn(4, 26).to(device)
x_c = torch.randn(4, 5).to(device)

final, aux1, aux2, aux3, attn = model_test(x_p, x_i, x_c)

print("Architecture test passed ✅")
print(f"  Final risk:      {final.shape}")
print(f"  Aux pathway:     {aux1.shape}")
print(f"  Aux immune:      {aux2.shape}")
print(f"  Aux clinical:    {aux3.shape}")
print(f"  Attention:       {attn.shape}")
print(f"  Attn sums:       {attn.sum(dim=1)}")

# Count parameters
total_params = sum(p.numel() for p in model_test.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"Patients per parameter: {478/total_params:.4f}")
print(f"Events per parameter:   {121/total_params:.4f}")

Architecture test passed ✅
  Final risk:      torch.Size([4, 1])
  Aux pathway:     torch.Size([4, 1])
  Aux immune:      torch.Size([4, 1])
  Aux clinical:    torch.Size([4, 1])
  Attention:       torch.Size([4, 3])
  Attn sums:       tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)

Total parameters: 8,709
Patients per parameter: 0.0549
Events per parameter:   0.0139


In [4]:
def cox_loss(risk_scores, times, events):
    order       = torch.argsort(times, descending=True)
    risk_scores = risk_scores[order].squeeze()
    events      = events[order]
    log_cumsum  = torch.logcumsumexp(risk_scores, dim=0)
    loss = -torch.mean((risk_scores - log_cumsum)[events.bool()])
    return loss


class SurvivalDataset(Dataset):
    def __init__(self, pathway, immune, clinical, times, events):
        self.pathway  = torch.FloatTensor(pathway)
        self.immune   = torch.FloatTensor(immune)
        self.clinical = torch.FloatTensor(clinical)
        self.times    = torch.FloatTensor(times)
        self.events   = torch.FloatTensor(events)
    def __len__(self): return len(self.times)
    def __getitem__(self, idx):
        return (self.pathway[idx], self.immune[idx],
                self.clinical[idx], self.times[idx], self.events[idx])


def train_deep_supervision(model, train_loader,
                            val_pathway, val_immune, val_clinical,
                            val_times, val_events,
                            epochs=300, patience=30, lr=0.001,
                            aux_weight=0.2, noise=0.05):
    """
    Train fusion model with deep supervision.
    
    Total loss = Final Cox loss
               + aux_weight × Pathway auxiliary Cox loss
               + aux_weight × Immune auxiliary Cox loss
               + aux_weight × Clinical auxiliary Cox loss
    
    aux_weight=0.2 means auxiliary heads contribute 20% each.
    This forces each stream to be independently predictive.
    """
    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=10, factor=0.5)

    best_val_ci  = 0
    best_weights = None
    patience_ctr = 0

    for epoch in range(epochs):
        model.train()
        for x_p, x_i, x_c, times, events in train_loader:
            x_p     = x_p.to(device)
            x_i     = x_i.to(device)
            x_c     = x_c.to(device)
            times   = times.to(device)
            events  = events.to(device)

            # Gaussian noise on molecular streams only
            x_p = x_p + torch.randn_like(x_p) * noise
            x_i = x_i + torch.randn_like(x_i) * noise

            optimizer.zero_grad()

            final, aux_p, aux_i, aux_c, _ = model(x_p, x_i, x_c)

            # Main loss
            loss_final = cox_loss(final, times, events)

            # Auxiliary losses — deep supervision
            loss_aux_p = cox_loss(aux_p, times, events)
            loss_aux_i = cox_loss(aux_i, times, events)
            loss_aux_c = cox_loss(aux_c, times, events)

            # Total loss
            total_loss = (loss_final
                         + aux_weight * loss_aux_p
                         + aux_weight * loss_aux_i
                         + aux_weight * loss_aux_c)

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # Validation on final risk only
        model.eval()
        with torch.no_grad():
            val_risk, _, _, _, _ = model(
                val_pathway.to(device),
                val_immune.to(device),
                val_clinical.to(device))
            val_risk = val_risk.squeeze().cpu().numpy()

        val_ci = concordance_index_censored(
            val_events.astype(bool), val_times, val_risk)[0]

        scheduler.step(-val_ci)

        if val_ci > best_val_ci:
            best_val_ci  = val_ci
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if patience_ctr >= patience:
            break

    model.load_state_dict(best_weights)
    return model, best_val_ci, epoch


# Test loss function
risk_t  = torch.randn(10, 1).to(device)
times_t = torch.randint(1, 1000, (10,)).float().to(device)
events_t = torch.randint(0, 2, (10,)).float().to(device)
loss_t  = cox_loss(risk_t, times_t, events_t)

print("Cox loss test ✅")
print("SurvivalDataset defined ✅")
print("train_deep_supervision defined ✅")
print(f"Test loss value: {loss_t.item():.4f}")

Cox loss test ✅
SurvivalDataset defined ✅
train_deep_supervision defined ✅
Test loss value: 1.6496


In [5]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_cindex       = []
fold_attn_weights = []

# Combined immune = raw fractions + ratios
immune_combined = pd.concat([immune, immune_ratios], axis=1)
print(f"Combined immune features: {immune_combined.shape[1]} (22 fractions + 4 ratios)")

print(f"\nRunning leakage-free 5-fold CV — Deep Supervision Fusion")
print(f"Streams: Pathway(50) + Immune(26) + Clinical(5)")
print(f"Deep supervision aux_weight=0.2 on all 3 streams")
print(f"{'Fold':<6} {'Best Epoch':<12} {'Test C-index':<12}")
print("-" * 32)

for fold, (train_idx, test_idx) in enumerate(kf.split(pathway_scores, y['event']), 1):

    # Split all streams
    p_train, p_test = pathway_scores.iloc[train_idx], pathway_scores.iloc[test_idx]
    i_train, i_test = immune_combined.iloc[train_idx], immune_combined.iloc[test_idx]
    c_train, c_test = clinical_features.iloc[train_idx], clinical_features.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    times_train  = y_train['time'].copy()
    events_train = y_train['event'].copy()
    times_test   = y_test['time'].copy()
    events_test  = y_test['event'].copy()

    # Scale on training only
    scaler_p = StandardScaler()
    scaler_i = StandardScaler()
    scaler_c = StandardScaler()

    p_train_s = scaler_p.fit_transform(p_train)
    p_test_s  = scaler_p.transform(p_test)
    i_train_s = scaler_i.fit_transform(i_train)
    i_test_s  = scaler_i.transform(i_test)
    c_train_s = scaler_c.fit_transform(c_train)
    c_test_s  = scaler_c.transform(c_test)

    # Validation set (20% of training)
    val_size     = int(0.2 * len(train_idx))
    val_pathway  = torch.FloatTensor(p_train_s[:val_size])
    val_immune   = torch.FloatTensor(i_train_s[:val_size])
    val_clinical = torch.FloatTensor(c_train_s[:val_size])
    val_times    = times_train[:val_size].copy()
    val_events   = events_train[:val_size].copy()

    # Dataset and loader
    train_ds = SurvivalDataset(
        p_train_s, i_train_s, c_train_s,
        times_train.copy(), events_train.copy())
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

    # Train model
    model = DeepSupervisionFusion(
        pathway_dim=50, immune_dim=26, clinical_dim=5).to(device)

    model, best_val_ci, best_epoch = train_deep_supervision(
        model, train_loader,
        val_pathway, val_immune, val_clinical,
        val_times, val_events,
        epochs=300, patience=30, lr=0.001,
        aux_weight=0.2, noise=0.05)

    # Evaluate on test fold
    model.eval()
    with torch.no_grad():
        test_risk, _, _, _, test_attn = model(
            torch.FloatTensor(p_test_s).to(device),
            torch.FloatTensor(i_test_s).to(device),
            torch.FloatTensor(c_test_s).to(device))

    test_risk = test_risk.squeeze().cpu().numpy()
    test_attn = test_attn.cpu().numpy()

    ci_test = concordance_index_censored(
        events_test.astype(bool), times_test, test_risk)[0]

    fold_cindex.append(ci_test)
    fold_attn_weights.append(test_attn)

    print(f"{fold:<6} {best_epoch:<12} {ci_test:.4f}")

print("-" * 32)
print(f"\nDeep Supervision Fusion C-index: {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")
print(f"\nFull comparison:")
print(f"  Cox Clinical baseline:    0.700")
print(f"  Gene ensemble (NB11c):    0.702 ± 0.057")
print(f"  Pathway XGBoost:          0.640 ± 0.075")
print(f"  Deep Supervision Fusion:  {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")

# Attention weights
stream_names = ['Pathway', 'Immune', 'Clinical']
all_weights  = np.concatenate(fold_attn_weights, axis=0)
mean_weights = all_weights.mean(axis=0)
print(f"\nMean attention weights:")
for name, w in zip(stream_names, mean_weights):
    print(f"  {name:<12} {w*100:.1f}%")

Combined immune features: 26 (22 fractions + 4 ratios)

Running leakage-free 5-fold CV — Deep Supervision Fusion
Streams: Pathway(50) + Immune(26) + Clinical(5)
Deep supervision aux_weight=0.2 on all 3 streams
Fold   Best Epoch   Test C-index
--------------------------------
1      121          0.6024
2      157          0.5923
3      122          0.6003
4      134          0.6205
5      117          0.7306
--------------------------------

Deep Supervision Fusion C-index: 0.629 ± 0.052

Full comparison:
  Cox Clinical baseline:    0.700
  Gene ensemble (NB11c):    0.702 ± 0.057
  Pathway XGBoost:          0.640 ± 0.075
  Deep Supervision Fusion:  0.629 ± 0.052

Mean attention weights:
  Pathway      47.2%
  Immune       27.1%
  Clinical     25.7%


In [9]:
from sklearn.svm import NuSVR
from sklearn.preprocessing import normalize
from scipy.optimize import nnls
import GEOparse
from datetime import datetime

# Load GSE68465 — reuse from NB18 if already in memory
# otherwise reload
print("Loading GSE68465...")
gse = GEOparse.get_GEO(geo="GSE68465",
                        destdir=f'{base}/data/external/',
                        silent=True)

gsm_data = {}
for gsm_name, gsm in gse.gsms.items():
    if gsm.table is not None and len(gsm.table) > 0:
        gsm_data[gsm_name] = gsm.table.set_index('ID_REF')['VALUE']

expr_raw_ext = pd.DataFrame(gsm_data).T
gpl = gse.gpls['GPL96']
probe_to_gene = gpl.table.set_index('ID')['Gene Symbol'].dropna()
probe_to_gene = probe_to_gene[probe_to_gene != '']

expr_filtered = expr_raw_ext[[c for c in expr_raw_ext.columns
                               if c in probe_to_gene.index]]
expr_filtered.columns = [probe_to_gene[c] for c in expr_filtered.columns]
expr_filtered = expr_filtered.astype(float)
expr_filtered = expr_filtered.T.groupby(level=0).mean().T
expr_log2_ext = np.log2(expr_filtered + 1)

# Pathway scores
pathway_scores_ext = compute_pathway_scores(expr_log2_ext, hallmark)

# Clinical + survival
survival_records = []
for gsm_name, gsm in gse.gsms.items():
    record = {'sample_id': gsm_name}
    for c in gsm.metadata.get('characteristics_ch1', []):
        if ':' in c:
            key, val = c.split(':', 1)
            record[key.strip()] = val.strip()
    survival_records.append(record)

survival_df = pd.DataFrame(survival_records).set_index('sample_id')

def parse_ptnm_stage(s):
    s = str(s).strip()
    try:
        n = int(s[2]) if 'N' in s and s[2].isdigit() else 0
        t = int(s[5]) if 'T' in s and s[5].isdigit() else 1
        if n == 0 and t == 1: return 'Stage I'
        elif n == 0 and t == 2: return 'Stage II'
        elif n == 1 and t in [1,2]: return 'Stage II'
        elif n == 2 or t in [3,4]: return 'Stage III'
        else: return 'Stage I'
    except: return 'Unknown'

survival_df['stage_group']   = survival_df['disease_stage'].apply(parse_ptnm_stage)
survival_df['survival_days'] = pd.to_numeric(
    survival_df['months_to_last_contact_or_death'], errors='coerce') * 30.44
survival_df = survival_df[survival_df['survival_days'].notna()]
survival_df = survival_df[survival_df['vital_status'].isin(['Alive', 'Dead'])]

common_ext = pathway_scores_ext.index.intersection(survival_df.index)
pathway_scores_ext = pathway_scores_ext.loc[common_ext]
survival_df        = survival_df.loc[common_ext]

# Clinical features
age_ext    = pd.to_numeric(survival_df['age'], errors='coerce').fillna(65)
gender_ext = (survival_df['Sex'] == 'Male').astype(float)
stage_II   = (survival_df['stage_group'] == 'Stage II').astype(float)
stage_III  = (survival_df['stage_group'] == 'Stage III').astype(float)
stage_IV   = (survival_df['stage_group'] == 'Stage IV').astype(float)

clinical_ext = pd.DataFrame({
    'age':             age_ext.values,
    'gender':          gender_ext.values,
    'stage_Stage II':  stage_II.values,
    'stage_Stage III': stage_III.values,
    'stage_Stage IV':  stage_IV.values
}, index=common_ext)

# CIBERSORT
lm22 = pd.read_csv(f'{base}/data/external/LM22.txt', sep='\t', index_col=0)
common_lm22   = lm22.index.intersection(expr_log2_ext.columns)
expr_lm22_ext = expr_log2_ext.loc[common_ext][common_lm22]
lm22_common   = lm22.loc[common_lm22]

def run_cibersort_single(patient_expr, lm22_matrix):
    expr_linear = (2 ** patient_expr.values) - 1
    expr_linear = np.clip(expr_linear, 0, 1e6)
    lm22_linear = (2 ** lm22_matrix.values) - 1
    lm22_linear = np.clip(lm22_linear, 0, 1e6)
    lm22_norm   = normalize(lm22_linear, axis=0)
    expr_norm   = normalize(expr_linear.reshape(1, -1))[0]
    best_nu = 0.5; best_error = np.inf
    for nu in [0.25, 0.5, 0.75]:
        try:
            svr = NuSVR(nu=nu, kernel='linear', C=1.0)
            svr.fit(lm22_norm, expr_norm)
            error = np.mean((lm22_norm @ svr.coef_[0] - expr_norm)**2)
            if error < best_error:
                best_error = error; best_nu = nu
        except: continue
    try:
        svr = NuSVR(nu=best_nu, kernel='linear', C=1.0)
        svr.fit(lm22_norm, expr_norm)
        raw_weights = svr.coef_[0]
    except:
        raw_weights = np.zeros(lm22_matrix.shape[1])
    clipped = np.maximum(raw_weights, 0)
    if clipped.sum() == 0:
        clipped, _ = nnls(lm22_norm, expr_norm)
        clipped = np.maximum(clipped, 0)
    total = clipped.sum()
    final = clipped / total if total > 0 else np.ones(len(clipped)) / len(clipped)
    return dict(zip(lm22_matrix.columns, final))

print(f"\nRunning CIBERSORT on {len(expr_lm22_ext)} patients...")
results_ext = {}
for i, pid in enumerate(expr_lm22_ext.index):
    results_ext[pid] = run_cibersort_single(expr_lm22_ext.loc[pid], lm22_common)
    if (i+1) % 100 == 0 or i == 0:
        print(f"  {i+1}/{len(expr_lm22_ext)} [{datetime.now().strftime('%H:%M:%S')}]")

immune_ext = pd.DataFrame(results_ext).T

# Immune ratios for external
immune_ratios_ext = pd.DataFrame({
    'CD8_Treg_ratio': (immune_ext['T cells CD8'] /
                      (immune_ext['T cells regulatory (Tregs)'] + 1e-8)),
    'M1_M2_ratio':    (immune_ext['Macrophages M1'] /
                      (immune_ext['Macrophages M2'] + 1e-8)),
    'NK_act_ratio':   (immune_ext['NK cells activated'] /
                      (immune_ext['NK cells resting'] + 1e-8)),
    'CD8_M2_ratio':   (immune_ext['T cells CD8'] /
                      (immune_ext['Macrophages M2'] + 1e-8)),
}, index=common_ext)

immune_combined_ext = pd.concat([immune_ext, immune_ratios_ext], axis=1)

print(f"\nExternal data ready:")
print(f"  Patients:  {len(common_ext)}")
print(f"  Pathways:  {pathway_scores_ext.shape[1]}")
print(f"  Immune:    {immune_combined_ext.shape[1]}")
print(f"  Clinical:  {clinical_ext.shape[1]}")

Loading GSE68465...

Running CIBERSORT on 442 patients...
  1/442 [01:22:50]
  100/442 [01:22:53]
  200/442 [01:22:56]
  300/442 [01:22:59]
  400/442 [01:23:03]

External data ready:
  Patients:  442
  Pathways:  50
  Immune:    26
  Clinical:  5


In [10]:
# Train final deep supervision model on ALL TCGA data
# Scale on full training data
scaler_p_final = StandardScaler()
scaler_i_final = StandardScaler()
scaler_c_final = StandardScaler()

p_train_final = scaler_p_final.fit_transform(pathway_scores)
i_train_final = scaler_i_final.fit_transform(immune_combined)
c_train_final = scaler_c_final.fit_transform(clinical_features)

# Validation set for early stopping (20% of training)
val_size     = int(0.2 * len(pathway_scores))
val_pathway  = torch.FloatTensor(p_train_final[:val_size])
val_immune   = torch.FloatTensor(i_train_final[:val_size])
val_clinical = torch.FloatTensor(c_train_final[:val_size])
val_times    = y['time'][:val_size].copy()
val_events   = y['event'][:val_size].copy()

train_ds_final = SurvivalDataset(
    p_train_final, i_train_final, c_train_final,
    y['time'].copy(), y['event'].copy())
train_loader_final = DataLoader(train_ds_final, batch_size=32, shuffle=True)

print("Training final deep supervision model on all 478 TCGA patients...")
model_final = DeepSupervisionFusion(
    pathway_dim=50, immune_dim=26, clinical_dim=5).to(device)

model_final, best_val_ci, best_epoch = train_deep_supervision(
    model_final, train_loader_final,
    val_pathway, val_immune, val_clinical,
    val_times, val_events,
    epochs=300, patience=30, lr=0.001,
    aux_weight=0.2, noise=0.05)

print(f"Training complete ✅ (best epoch: {best_epoch}, val C-index: {best_val_ci:.3f})")

# Scale external data using training scalers
p_ext_s = scaler_p_final.transform(pathway_scores_ext)
i_ext_s = scaler_i_final.transform(immune_combined_ext)
c_ext_s = scaler_c_final.transform(clinical_ext)

# Predict on GSE68465
model_final.eval()
with torch.no_grad():
    ext_risk, _, _, _, ext_attn = model_final(
        torch.FloatTensor(p_ext_s).to(device),
        torch.FloatTensor(i_ext_s).to(device),
        torch.FloatTensor(c_ext_s).to(device))

ext_risk = ext_risk.squeeze().cpu().numpy()
ext_attn = ext_attn.cpu().numpy()

# Survival labels
y_ext = np.array(
    [(vs == 'Dead', float(t)) for vs, t in
     zip(survival_df['vital_status'], survival_df['survival_days'])],
    dtype=[('event', bool), ('time', float)])

ci_ext = concordance_index_censored(
    y_ext['event'].astype(bool),
    y_ext['time'], ext_risk)[0]

print(f"\n{'='*55}")
print(f"GSE68465 EXTERNAL VALIDATION — DEEP SUPERVISION FUSION")
print(f"{'='*55}")
print(f"Patients:        {len(y_ext)}")
print(f"Events:          {y_ext['event'].sum()} ({y_ext['event'].mean()*100:.1f}%)")
print(f"C-index:         {ci_ext:.3f}")
print(f"{'='*55}")
print(f"\nFull comparison on GSE68465:")
print(f"  Gene model (NB11c):          0.637")
print(f"  Pathway XGBoost (NB18):      0.654")
print(f"  Deep Supervision Fusion:     {ci_ext:.3f}")
print(f"\nMean attention weights (external):")
stream_names = ['Pathway', 'Immune', 'Clinical']
for name, w in zip(stream_names, ext_attn.mean(axis=0)):
    print(f"  {name:<12} {w*100:.1f}%")

Training final deep supervision model on all 478 TCGA patients...
Training complete ✅ (best epoch: 185, val C-index: 0.989)

GSE68465 EXTERNAL VALIDATION — DEEP SUPERVISION FUSION
Patients:        442
Events:          236 (53.4%)
C-index:         0.619

Full comparison on GSE68465:
  Gene model (NB11c):          0.637
  Pathway XGBoost (NB18):      0.654
  Deep Supervision Fusion:     0.619

Mean attention weights (external):
  Pathway      44.5%
  Immune       30.6%
  Clinical     24.9%
